# Pushing markers

`boris_push` advances the markers of a `ParticleStore` by one time step in electric and magnetic fields given at the markers: gathered from a grid (for example with mpifield) or prescribed. It is the Boris scheme, compiled with Pyccel (CUDA for CuPy arrays), for 1, 2 or 3 position components with three velocity components.

This tutorial follows an electron gyrating in a magnetic field, adds an electric field to get the E×B drift, and pushes the same markers in 1D3V, 2D3V and 3D3V.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mpiparticles import ParticleStore, boris_push

ELECTRON = -1.602176634e-19 / 9.1093837015e-31  # charge over mass [C/kg]

## Gyration

An electron in a uniform magnetic field `B = 10 mT` along z gyrates at the cyclotron frequency $\omega_c = |q|B/m$. The fields are arrays `(3, length)`, one column per stored marker, or `(3,)` for the same field everywhere. The step here is $\omega_c \Delta t = 0.2$.

In [ ]:
b = np.array([0.0, 0.0, 1e-2])
omega_c = abs(ELECTRON) * b[2]
dt = 0.2 / omega_c

store = ParticleStore(ndim=2, nvel=3)
store.add(np.zeros((2, 1)), np.array([[1e5], [0.0], [0.0]]))  # 100 km/s along x
orbit = [store.positions[:, 0].copy()]
for _ in range(int(2 * np.pi / 0.2)):
    boris_push(store, [0.0, 0.0, 0.0], b, dt, ELECTRON)
    orbit.append(store.positions[:, 0].copy())
orbit = np.array(orbit) * 1e3  # mm

speed = np.linalg.norm(store.velocities[:, 0])
print(f"gyroradius {1e5 / omega_c * 1e3:.3f} mm, speed after one turn {speed:.6e} m/s")

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(orbit[:, 0], orbit[:, 1], "o-", ms=3)
ax.set_aspect("equal")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("An electron gyrating about B along z")
plt.show()

The speed is kept exactly in a pure magnetic field: the Boris rotation only turns the velocity. It turns it by $2\arctan(\omega_c\Delta t/2)$ per step instead of $\omega_c\Delta t$, a small phase lag at this step.

## The E×B drift

With an electric field `E = 1 kV/m` along y the guiding centre drifts with $\mathbf{E}\times\mathbf{B}/B^2 = 100$ km/s along x, whatever the charge. An electron starting at rest traces a cycloid.

In [ ]:
e = np.array([0.0, 1e3, 0.0])
store = ParticleStore(ndim=2, nvel=3)
store.add(np.zeros((2, 1)), np.zeros((3, 1)))
path = []
steps = 3 * int(2 * np.pi / 0.2)
for _ in range(steps):
    boris_push(store, e, b, dt, ELECTRON)
    path.append(store.positions[:, 0].copy())
path = np.array(path) * 1e3

drift = np.cross(e, b) / b.dot(b)
print(
    f"E x B / B^2 = {drift[0] / 1e3:.0f} km/s along x; "
    f"mean speed of the electron {store.positions[0, 0] / (steps * dt) / 1e3:.1f} km/s"
)

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(path[:, 0], path[:, 1])
ax.set_aspect("equal")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("E along y, B along z: the cycloid of an electron")
plt.show()

## 1D3V, 2D3V and 3D3V

The same call pushes stores with one, two or three position components; the velocity always has three, and the first `ndim` of them move the positions. A 1D3V store keeps the full gyration in velocity space while only `x` moves.

In [ ]:
rng = np.random.default_rng(1)
for ndim in (1, 2, 3):
    store = ParticleStore(ndim=ndim, nvel=3)
    store.add(np.zeros((ndim, 1000)), rng.normal(0.0, 1e5, size=(3, 1000)))
    energy = np.sum(store.velocities[:, :1000] ** 2)
    for _ in range(50):
        boris_push(store, [0.0, 0.0, 0.0], [3e-3, 0.0, 1e-2], dt, ELECTRON)
    change = np.sum(store.velocities[:, :1000] ** 2) / energy - 1
    print(
        f"{ndim}D3V: positions {store.positions[:, :1000].shape}, "
        f"kinetic energy changed by {change:.1e}"
    )

The fields at the markers come from the caller: with a grid field, gather them with mpifield first (`Field.gather` at `store.positions[:, :store.length]`), then push. Dead markers keep their columns until `compact` and are pushed with the rest; spare capacity past `length` is not touched.